# Mahabharata NER — Kaggle runner

Runs `mahabharata_ner_final.py` unchanged and shows **each model individually** (leaderboard + per-class tables + figures).

## One-time setup before you run
1. **Add Data** (right panel) → upload a dataset that contains **`TheMahabharata.pdf`** *and* **`mahabharata_ner_final.py`** (one dataset with both, or two datasets — the notebook auto-discovers them under `/kaggle/input/`).
2. **Settings → Internet → ON** (needed to pip-install `sklearn-crfsuite`, `seqeval`, `lime`, `pymupdf` and download the spaCy model).
3. To run the long full pipeline headless within the 12h limit, use **Save Version → Save & Run All (Commit)**.

> The script is already Kaggle-aware: it globs `/kaggle/input/*/*.pdf` for the corpus and writes to `./outputs` (= `/kaggle/working/outputs`, which we set explicitly below).

## 1. Install the dependencies Kaggle doesn't preinstall
`numpy / pandas / scipy / scikit-learn / matplotlib / spacy / xgboost / shap` ship with the Kaggle image. We add the rest.

In [ ]:
import sys, subprocess

def sh(*cmd):
    print(">", *cmd, flush=True)
    subprocess.run(list(cmd), check=True)

sh(sys.executable, "-m", "pip", "install", "-q",
   "sklearn-crfsuite", "seqeval", "lime", "pymupdf")
sh(sys.executable, "-m", "spacy", "download", "en_core_web_sm")
print("\nDependencies ready.")

## 2. Locate the PDF + script, set the output directory
Set `NER_FAST=1` in the cell below for a ~3 min end-to-end smoke test first; remove it for the full run.

In [ ]:
import os, glob

pdfs = sorted(glob.glob("/kaggle/input/**/*.pdf", recursive=True))
scripts = sorted(glob.glob("/kaggle/input/**/mahabharata_ner_final.py", recursive=True))
print("PDF(s):   ", pdfs)
print("Script(s):", scripts)
assert pdfs, "No PDF under /kaggle/input — Add Data with TheMahabharata.pdf."
assert scripts, "mahabharata_ner_final.py not found under /kaggle/input — add it as a dataset."
SCRIPT = scripts[0]

os.environ["NER_OUT"] = "/kaggle/working/outputs"   # persisted + downloadable
# os.environ["NER_FAST"] = "1"                       # uncomment for a quick smoke test
print("\nUsing script:", SCRIPT)
print("Output dir:  ", os.environ["NER_OUT"], "| fast_mode =", os.environ.get("NER_FAST", "0"))

## 3. Run the full pipeline
`main()` executes on import, so running the file runs the whole pipeline (corpus → features → every model → CRF → stacking → SHAP/LIME → diagrams). Logs stream below.

In [ ]:
import runpy
runpy.run_path(SCRIPT, run_name="__main__")

## 4. Each model, individually
**Leaderboard** (every model, ranked by entity-span micro-F1) → then the **per-class P/R/F1 table for each model** → then the **feature-representation comparison** (same balanced-LR classifier, different feature sets).

In [ ]:
import pandas as pd, glob, os
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

run_dir = max(glob.glob("/kaggle/working/outputs/run_*"), key=os.path.getmtime)
print("Run directory:", run_dir)

print("\n=============== MODEL LEADERBOARD (PRIMARY = entity-span micro-F1) ===============")
lb = pd.read_csv(f"{run_dir}/tables/model_comparison.csv")
display(lb)

for f in sorted(glob.glob(f"{run_dir}/tables/per_class_*.csv")):
    model = os.path.basename(f)[len("per_class_"):-4]
    print(f"\n=============== {model} — per-class precision / recall / F1 ===============")
    display(pd.read_csv(f, index_col=0))

print("\n=============== FEATURE-REPRESENTATION COMPARISON (fixed balanced LR) ===============")
display(pd.read_csv(f"{run_dir}/tables/vectorizer_comparison.csv"))

## 5. Figures (confusion matrix, SHAP, ablation, diagrams)

In [ ]:
from IPython.display import Image, display
import glob, os
for img in sorted(glob.glob(f"{run_dir}/figures/*.png")):
    print(os.path.basename(img))
    display(Image(filename=img))

## 6. Bundle every artifact for download
Grab the zip from the **Output** tab (or the Data panel of the committed version).

In [ ]:
import shutil
zip_path = shutil.make_archive("/kaggle/working/mahabharata_outputs", "zip", run_dir)
print("Wrote:", zip_path)